# 实验三：pandas 数据处理基础

<table style="margin: 0 auto; width: 30%; border-collapse: collapse; border: 1px solid black;" data-id="student-info">  
    <colgroup>  
        <col style="width: 35%;">  
        <col style="width: 65%;">  
    </colgroup>  
    <tr>  
        <td style="border: 1px solid black;">班级</td> <td style="border: 1px solid black;">25智能01</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">学号</td> <td style="border: 1px solid black;">未填写</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">姓名</td> <td style="border: 1px solid black;">未填写</td>  
    </tr>  
    <tr>  
        <td style="border: 1px solid black;">Email</td> <td style="border: 1px solid black;">123456@qq.com</td>  
    </tr>  
</table>

## 实验目标

围绕零售数据练习：数据选择与排序、缺失与重复处理、字符串清洗、分类分箱、连接与拼接、分组聚合、宽长表转换。

先读每节的小例子，再把同样的方法用于对应习题。实验主要参考《Python for Data Analysis》第3版第5、7、8章。

## 术语速查

常用术语见下表，按需查阅；代码中的类型、函数和属性名保留原文。

| 中文术语 | 英文名称 | pandas API 或说明 |
|---|---|---|
| 序列 / 数据框 | Series / DataFrame | pandas 的一维 / 二维带标签数据结构 |
| 索引 / 行标签 / 列标签 | Index / Row Label / Column Label | 数据选择与对齐的基础 |
| 标签索引 / 位置索引 | Label-based Indexing / Positional Indexing | `loc` / `iloc` |
| 布尔索引（条件筛选） | Boolean Indexing (Conditional Filtering) | 用布尔条件选择行 |
| 副本 / 设置索引 / 重置索引 | Copy / Set Index / Reset Index | `copy()` / `set_index()` / `reset_index()` |
| 排序 | Sorting | `sort_values()` |
| 缺失值 / 填充缺失值 | Missing Value / Missing-value Imputation | `isna()` / `fillna()` |
| 重复值 / 去重 | Duplicate / Deduplication | `duplicated()` / `drop_duplicates()` |
| 数据类型转换 | Data Type Conversion | `astype()` / `to_datetime()` |
| 向量化字符串操作 | Vectorized String Operation | `.str` 访问器及 `strip()`、`lower()` 等 |
| 分类数据 / 分箱 | Categorical Data / Binning | `Categorical` / `cut()` |
| 表连接 / 合并键 | Table Join (Merge) / Join Key | `merge()` |
| 拼接 / 轴 | Concatenation / Axis | `concat()` |
| 分组 / 聚合 | Grouping / Aggregation | `groupby()` / `agg()` |
| 拆分—应用—合并 | Split–Apply–Combine | 分组计算的工作模式 |
| 宽表 / 长表 | Wide Format / Long Format | 两种常见数据布局 |
| 数据重塑 | Data Reshaping | `melt()`、`pivot()`、`stack()`、`unstack()` |
| CSV 文件 | Comma-Separated Values (CSV) File | `read_csv()` |
| 自动对齐 | Automatic Alignment | pandas 按索引或列名对齐数据 |
| 内连接 / 左连接 | Inner Join / Left Join | `how='inner'` / `how='left'` |
| 连接基数校验 | Join Cardinality Validation | `one_to_one`、`one_to_many`、`many_to_one`、`many_to_many` |
| 命名聚合 / 唯一值计数 | Named Aggregation / Distinct Count | `.agg()` / `nunique()` |
| 日期时间 | Datetime | `to_datetime()` |

## Notebook 使用说明

1. 选择项目 `venv` 对应的 Python 内核，并填写顶部学生信息。
2. 按顺序运行单元格；修改后先保存，再运行对应测试。
3. 若结果与代码不一致，重启内核并从头运行；仍报错时，先检查第一个错误。

## 实验注意事项

- 只在指定区域补全代码；保留函数名、参数、变量名及 `UNQ_C`、`GRADED FUNCTION` 标记。
- 不改测试、评分和提交代码，不增删代码单元格。`...` 是待补全部分；提示行数仅供参考。
- 完成一题后保存并运行对应测试，再继续下一题。
- 独立完成实验，禁止抄袭；抄袭成绩记0分。

## 1. 实验准备

In [1]:
# 导入依赖并初始化自动测试
%matplotlib inline

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython import get_ipython

ip = get_ipython()
notebook_file = None
if ip is not None:
    notebook_file = ip.user_ns.get('__vsc_ipynb_file__')

if notebook_file is None:
    candidates = list(Path.cwd().glob('**/实验3-pandas.ipynb'))
    notebook_file = str(candidates[0]) if candidates else None

notebook_dir = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
project_root = notebook_dir.parent
data_dir = notebook_dir / 'data'

sys.path.append(str(project_root / 'tests'))
sys.path.append(str(project_root / 'util'))

from test_suite3 import TestSuite3
from notebook_info_extractor import extract_from_ipynb, is_valid_email

if notebook_file is None:
    raise FileNotFoundError('无法定位实验3-pandas.ipynb，请从项目目录中打开并运行本 Notebook。')

testsuite3 = TestSuite3()
stu_info = extract_from_ipynb(notebook_file)

if is_valid_email(stu_info.get('email')):
    print(f"stu_info: {stu_info}")
else:
    print(
        'Email地址无效，请在Notebook顶部的学生信息表中重新输入有效的Email地址，'
        '保存Notebook后重新运行本单元格。'
    )

stu_info: {'class_id': '25智能01', 'student_id': '未填写', 'name': '未填写', 'email': '123456@qq.com'}


### 1.1 实验数据

业务讲解和习题数据来自 `data` 目录中的 CSV 文件：

| 文件 | 含义 | 主要字段 |
|---|---|---|
| `products.csv` | 商品主数据 | 商品编号、类别、价格、状态 |
| `orders.csv` | 原始订单 | 日期、客户、商品、数量、折扣 |
| `customers_dirty.csv` | 待清洗客户数据 | 姓名、城市、客户类型、邮箱 |
| `inventory_shanghai.csv` | 上海仓库存 | 商品、仓库、库存量 |
| `inventory_beijing.csv` | 北京仓库存 | 商品、仓库、库存量 |
| `monthly_sales_wide.csv` | 月度销售宽表 | 商品及各月份销售额 |

这些是教学用模拟数据；独立小例子直接在代码中创建表。

In [2]:
products = pd.read_csv(data_dir / 'products.csv')
orders = pd.read_csv(data_dir / 'orders.csv')
customers_raw = pd.read_csv(data_dir / 'customers_dirty.csv')
inventory_shanghai = pd.read_csv(data_dir / 'inventory_shanghai.csv')
inventory_beijing = pd.read_csv(data_dir / 'inventory_beijing.csv')
monthly_sales_wide = pd.read_csv(data_dir / 'monthly_sales_wide.csv')

pd.DataFrame({
    '数据表': ['products', 'orders', 'customers_raw', 'inventory_shanghai',
             'inventory_beijing', 'monthly_sales_wide'],
    '行数': [len(products), len(orders), len(customers_raw), len(inventory_shanghai),
           len(inventory_beijing), len(monthly_sales_wide)],
    '列数': [products.shape[1], orders.shape[1], customers_raw.shape[1],
           inventory_shanghai.shape[1], inventory_beijing.shape[1], monthly_sales_wide.shape[1]],
})

,数据表,行数,列数
0,products,8,5
1,orders,13,6
2,customers_raw,8,5
3,inventory_shanghai,5,3
4,inventory_beijing,5,3
5,monthly_sales_wide,7,4


## 2. 数据选择

`Series` 是带索引的一维数据，`DataFrame` 是二维表；索引还用于对齐运算。

| 写法 | 作用 |
|---|---|
| `df['price']` | 取单列，通常返回Series |
| `df[['name','price']]` | 取多列，返回DataFrame |
| `df.loc[行标签,列标签]` | 按标签选择 |
| `df.iloc[行位置,列位置]` | 按从0开始的位置选择 |
| `df[布尔条件]` | 筛选行 |

多条件用带括号的 `&`（并且）、`|`（或者）、`~`（取反），不用 `and/or`。`eq/ne/lt/le/gt/ge` 分别等价于 `==/!=/</<=/>/>=`。

下面行标签为 `r,s,t`，位置仍为0、1、2。

```python
import pandas as pd
selection_demo = pd.DataFrame({
    'name': ['Pen', 'Bag', 'Book'],
    'price': [8, 150, 120],
    'stock': [20, 0, 5],
}, index=['r', 's', 't'])
print(type(selection_demo['price']).__name__)            # Series
print(type(selection_demo[['name', 'price']]).__name__) # DataFrame
print(selection_demo.iloc[0]['name'])                  # Pen：第一行
print(selection_demo.loc['s', 'name'])                 # Bag：标签s的行
```

```python
available_demo = selection_demo[
    (selection_demo['price'] > 100) & (selection_demo['stock'] > 0)
]
print(available_demo['name'].tolist())  # ['Book']
```

In [3]:
products.head(4)

,product_id,product_name,category,unit_price,status
0,P001,Wireless Mouse,Electronics,89,active
1,P002,Mechanical Keyboard,Electronics,329,active
2,P003,USB-C Hub,Electronics,199,active
3,P004,Notebook,Stationery,18,active


In [4]:
# loc 按标签或条件选择；多个条件必须分别加括号
products.loc[
    (products['status'].eq('active')) & (products['unit_price'].le(200)),
    ['product_id', 'product_name', 'category', 'unit_price'],
]

,product_id,product_name,category,unit_price
0,P001,Wireless Mouse,Electronics,89
2,P003,USB-C Hub,Electronics,199
3,P004,Notebook,Stationery,18
4,P005,Gel Pen Set,Stationery,25
6,P007,Water Bottle,Home,79


In [5]:
# iloc 按位置选择：前三行、前三列
products.iloc[:3, :3]

,product_id,product_name,category
0,P001,Wireless Mouse,Electronics
1,P002,Mechanical Keyboard,Electronics
2,P003,USB-C Hub,Electronics


### 2.1 副本、索引与排序

| 方法 | 作用 |
|---|---|
| `copy()` | 创建副本，避免给输入表直接赋值 |
| `set_index('id')` | 把业务键变为行索引，默认移除原列 |
| `sort_values('price',ascending=False)` | 价格降序排序 |
| `reset_index(drop=True)` | 重建连续索引，不保留旧索引列 |

多列排序可用 `sort_values(['price','name'],ascending=[False,True])`：先按价格降序，同价再按名称升序。排序不会自动重建索引。

下面的副本可独立新增普通列；若单元格存放可变Python对象，`copy()` 不会递归复制对象内部。

```python
import pandas as pd
original_demo = pd.DataFrame({'id': ['A', 'B'], 'price': [8, 15]})
copy_demo = original_demo.copy()
copy_demo['discount'] = 0.1
assert 'discount' not in original_demo.columns
indexed_demo = copy_demo.set_index('id')
print(indexed_demo.loc['B', 'price'])  # 15
sorted_demo = copy_demo.sort_values('price', ascending=False).reset_index(drop=True)
print(sorted_demo['id'].tolist())     # ['B', 'A']：降序
print(sorted_demo.index.tolist())     # [0, 1]：重新编号
```

In [6]:
# 复合排序示例：价格降序；价格相同时，商品名称升序
products_sort_example = (
    products.copy()
    .sort_values(
        by=['unit_price', 'product_name'],
        ascending=[False, True],
    )
    .set_index('product_id')
)
products_sort_example[['product_name', 'category', 'unit_price']].head()

,product_name,category,unit_price
product_id,,,
P002,Mechanical Keyboard,Electronics,329
P008,Laptop Stand,Electronics,219
P003,USB-C Hub,Electronics,199
P006,Desk Lamp,Home,159
P001,Wireless Mouse,Electronics,89


In [7]:
products[(products['status'].eq('active')) & (products['unit_price'].le(200))]

,product_id,product_name,category,unit_price,status
0,P001,Wireless Mouse,Electronics,89,active
2,P003,USB-C Hub,Electronics,199,active
3,P004,Notebook,Stationery,18,active
4,P005,Gel Pen Set,Stationery,25,active
6,P007,Water Bottle,Home,79,active


## 习题1：条件筛选与排序

完成 `select_large_discount_orders(orders, min_quantity, min_discount)`。

数据：`data/orders.csv`，已读入 `orders`。

1. 不修改输入 DataFrame；
2. 只保留 `quantity >= min_quantity` 且 `discount >= min_discount` 的订单；
3. 将 `order_id` 设置为索引；
4. 只返回 `order_date`、`customer_id`、`product_id`、`quantity`、`discount` 五列；
5. 先按 `quantity` 降序排列，数量相同时再按 `order_date` 升序排列。

In [8]:
# UNQ_C1（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: select_large_discount_orders

def select_large_discount_orders(orders, min_quantity, min_discount):
    """筛选达到数量和折扣下限的订单。"""
    # 完成提示：
    # 步骤1：分别对 quantity 和 discount 构造大于等于参数的布尔条件。
    # 步骤2：用 & 组合条件，并用 loc 筛选行和需要的列。
    # 步骤3：把 order_id 设置为索引。
    # 步骤4: 选择需要返回的列，一定要按题目要求的顺序返回列。
    # 步骤5：按 quantity 降序、order_date 升序进行复合排序。
    result = ...

    return result

In [9]:
# 测试习题1
testsuite3.test_select_large_discount_orders(select_large_discount_orders)

测试失败 test_select_large_discount_orders: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 3. 缺失、重复与类型

- `isna()/notna()` 检查缺失，`fillna()` 填补缺失；空字符串和空格不是缺失标记。
- `duplicated()` 返回重复标记，`drop_duplicates()` 返回去重后的表。
- `subset` 指判断重复的业务键；`keep='last'` 保留当前行顺序中的最后一条，不自动等于日期最新。
- `pd.to_datetime()` 转日期，`astype()` 转其他类型，`dtypes` 检查各列类型。

清洗前先明确业务规则：例如本实验把缺失折扣解释为0，同一订单编号的后记录视为修正版。

```python
import pandas as pd
missing_demo = pd.Series([5.0, None, 9.0])
print(missing_demo.isna().tolist())     # [False, True, False]
print(missing_demo.fillna(0).tolist())  # [5.0, 0.0, 9.0]
```

```python
duplicate_demo = pd.DataFrame({'id': ['A', 'A', 'B'], 'value': [2, 7, 4]})
print(duplicate_demo.duplicated('id', keep='last').tolist())
# [True, False, False]：标记不是最终保留的记录
latest_demo = duplicate_demo.drop_duplicates('id', keep='last')
print(latest_demo['value'].tolist())  # [7, 4]
print(pd.to_datetime(pd.Series(['2027-03-01'])).dt.year.tolist())  # [2027]
```

In [10]:
pd.DataFrame({
    '缺失值数量': orders.isna().sum(),
    '数据类型': orders.dtypes.astype(str),
})

,缺失值数量,数据类型
order_id,0,object
order_date,0,object
customer_id,0,object
product_id,0,object
quantity,0,int64
discount,3,float64


In [11]:
orders.loc[
    orders.duplicated(
        subset='order_id', keep=False
        )
    ].sort_values('order_id')

,order_id,order_date,customer_id,product_id,quantity,discount
5,O1006,2026-03-04,C005,P005,4,NaN
6,O1006,2026-03-04,C005,P005,5,0.05


In [12]:
# 清洗演示：使用副本，避免意外修改原始数据
orders_demo = orders.copy()

orders_demo['discount'] = orders_demo['discount'].fillna(0.0)

orders_demo = orders_demo.drop_duplicates(
    subset='order_id', keep='last'
    )

orders_demo['order_date'] = pd.to_datetime(orders_demo['order_date'])
orders_demo['quantity'] = orders_demo['quantity'].astype('int64')
orders_demo = orders_demo.sort_values('order_id').reset_index(drop=True)
orders_demo.head()

,order_id,order_date,customer_id,product_id,quantity,discount
0,O1001,2026-03-01,C001,P001,2,0.10
1,O1002,2026-03-01,C002,P002,1,0.00
2,O1003,2026-03-02,C003,P004,5,0.05
3,O1004,2026-03-03,C001,P003,1,0.15
4,O1005,2026-03-03,C004,P007,3,0.00


## 习题2：缺失值处理与去重

完成 `clean_customer_records(customers)`。

数据：`data/customers_dirty.csv`，已读入 `customers`。

1. 不修改输入 DataFrame；
2. 将 `email` 的缺失值填为 `'unknown'`；
3. 按 `customer_id` 删除重复记录，保留第一条记录；
4. 将 `segment` 转换为 pandas 的 `string` 类型；
5. 按 `customer_id` 降序排列并重置索引；
6. 保持原有列及列顺序不变。

In [13]:
# UNQ_C2（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: clean_customer_records

def clean_customer_records(customers):
    """清理客户记录中的缺失值、重复值和数据类型。"""
    # 完成提示：
    # 步骤1：使用 copy() 创建工作副本。
    # 步骤2：只对 email 列使用 fillna('unknown')。
    # 步骤3：按 customer_id 去重，并注意本题要求 keep='first'。
    # 步骤4：用 astype('string') 转换 segment。
    # 步骤5：按 customer_id 降序排序并 reset_index(drop=True)。
    # 易错提醒：本题不要求清洗字符串空格，也不应填补 city；保持列顺序不变。
    result = ...
    return result

In [14]:
# 测试习题2
testsuite3.test_clean_customer_records(clean_customer_records)

测试失败 test_clean_customer_records: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 4. 字符串清洗

`.str` 对整列执行字符串操作：`strip()` 去首尾空白，`lower()/upper()/title()` 改大小写，`contains()` 查模式，`replace()` 替换，`extract()` 提取。

空格字符串需先 `strip()`，再替换 `''`；缺失值另用 `fillna()` 处理。邮箱通常统一为小写。

`str.contains(...,na=False)` 把缺失视为不匹配。默认区分大小写、支持正则；匹配普通字面文本时可指定 `regex=False`。

```python
import pandas as pd
cities_demo = pd.Series(['  Chengdu ', '   '])
print(cities_demo.str.strip().replace('', 'Unknown').tolist())
# ['Chengdu', 'Unknown']
print(pd.Series(['  USER@EXAMPLE.COM ']).str.strip().str.lower().tolist())
# ['user@example.com']
```

```python
texts_demo = pd.Series(['cat food', None, 'dog food'])
print(texts_demo.str.contains('cat', na=False).tolist())  # [True, False, False]
```

In [15]:
customers_raw[['customer_id', 'city', 'segment', 'email']]

,customer_id,city,segment,email
0,C001,shanghai,Student,ALICE@EXAMPLE.COM
1,C002,BEIJING,teacher,bob@example.com
2,C003,SHANGHAI,student,NaN
3,C004,guangzhou,corporate,old-diana@example.com
4,C004,Guangzhou,Corporate,diana@example.com
5,C005,NaN,student,evan@example.com
6,C006,beijing,Teacher,FANG@EXAMPLE.COM
7,C007,guangzhou,corporate,grace@example.com


In [16]:
customers_demo = customers_raw.drop_duplicates('customer_id', keep='last').copy()
customers_demo['customer_name'] = customers_demo['customer_name'].str.strip()
customers_demo['city'] = customers_demo['city'].fillna('').str.strip().replace('', 'Unknown').str.title()
customers_demo['segment'] = customers_demo['segment'].str.strip().str.lower()
customers_demo['email'] = customers_demo['email'].fillna('').str.strip().str.lower().replace('', 'unknown')
customers_demo = customers_demo.sort_values('customer_id').reset_index(drop=True)
customers_demo

,customer_id,customer_name,city,segment,email
0,C001,Alice Zhang,Shanghai,student,alice@example.com
1,C002,Bob Li,Beijing,teacher,bob@example.com
2,C003,Chen Wang,Shanghai,student,unknown
3,C004,Diana Liu,Guangzhou,corporate,diana@example.com
4,C005,Evan Zhou,Unknown,student,evan@example.com
5,C006,Fang Wu,Beijing,teacher,fang@example.com
6,C007,Grace He,Guangzhou,corporate,grace@example.com


## 习题3：向量化字符串清洗

完成 `standardize_product_text(products)`。

数据：`data/products.csv`，已读入 `products`。

1. 不修改输入 DataFrame；
2. 按 `product_id` 删除重复记录，保留第一条；
3. `product_name` 删除首尾空白并转换为标题格式；
4. `category` 删除首尾空白并转换为小写；
5. `status` 删除首尾空白并转换为大写；
6. 按 `product_name`、`product_id` 升序排列并重置索引；
7. 保持原有列及列顺序不变。

In [17]:
# UNQ_C3（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: standardize_product_text

def standardize_product_text(products):
    """统一商品名称、类别和状态的文本格式。"""
    # 完成提示：
    # 步骤1：创建副本，并按 product_id 去重，保留第一条。
    # 步骤2：对 product_name 依次使用 str.strip() 和 str.title()。
    # 步骤3：对 category 使用 str.strip() 和 str.lower()。
    # 步骤4：对 status 使用 str.strip() 和 str.upper()。
    # 步骤5：按 product_name、product_id 升序排序并重置索引。
    # 易错提醒：三个文本列的大小写规则不同，不要照搬客户字段的处理规则。
    result = ...
    return result

In [18]:
# 测试习题3
testsuite3.test_standardize_product_text(standardize_product_text)

测试失败 test_standardize_product_text: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 5. 分类与分箱

重复类别较少的文本可转为 `category`，用“类别+编码”保存。`cut` 按给定边界分箱，`qcut` 按样本分位数分箱，使各箱数量尽量接近，而非区间等宽。

| `cut` 参数 | 含义 |
|---|---|
| `bins` | 边界；4个边界通常形成3个区间 |
| `labels` | 各区间标签，数量应与区间数相同 |
| `right=True` | 包含右端点 |
| `include_lowest=True` | 第一区间也包含最低边界 |
| `ordered=True` | 类别有顺序 |

大量相同值可能让 `qcut` 的边界重合，需检查数据。用 `.dtype`、`.cat.categories`、`.cat.ordered` 检查分类结果。`assign` 返回含新列的新表，不原地修改输入。

```python
import pandas as pd
bands_demo = pd.cut([0, 50, 51, 100], bins=[0, 50, 100],
                    labels=['Low', 'High'], right=True, include_lowest=True)
print(bands_demo.tolist())  # ['Low', 'Low', 'High', 'High']
print(str(bands_demo.dtype))  # category
```

```python
quantiles_demo = pd.qcut(pd.Series(range(8)), q=2, labels=['Lower', 'Upper'])
print(quantiles_demo.value_counts().sort_index().tolist())  # [4, 4]
source_demo = pd.DataFrame({'quantity': [2, 3]})
assigned_demo = source_demo.assign(total=source_demo['quantity']*4)
print(assigned_demo['total'].tolist())  # [8, 12]
assert 'total' not in source_demo.columns
```

In [19]:
price_bins = [0, 100, 200, np.inf]
price_labels = ['低价', '中价', '高价']

products_with_band = products.assign(
    price_band=pd.cut(
        products['unit_price'],
        bins=price_bins,
        labels=price_labels,
        include_lowest=True,
        ordered=True,
    )
)
products_with_band[['product_id', 'unit_price', 'price_band']]

,product_id,unit_price,price_band
0,P001,89,低价
1,P002,329,高价
2,P003,199,中价
3,P004,18,低价
4,P005,25,低价
5,P006,159,中价
6,P007,79,低价
7,P008,219,高价


In [20]:
products_with_band['price_band'].dtype, products_with_band['price_band'].cat.categories

(CategoricalDtype(categories=['低价', '中价', '高价'], ordered=True, categories_dtype=object),
 Index(['低价', '中价', '高价'], dtype='object'))

## 习题4：分类数据与分箱

完成 `add_order_size_band(orders)`。

数据：`data/orders.csv`，已读入 `orders`。

1. 不修改输入 DataFrame；
2. 使用 `pd.cut` 新增有序分类列 `order_size`；
3. 分箱边界为 `[0, 1, 3, 正无穷]`；
4. 标签依次为 `'单件'`、`'小批'`、`'大批'`；
5. 包含最低值，且 `1` 属于单件、`3` 属于小批；
6. 返回原有全部列，并把 `order_size` 放在最后。

In [21]:
# UNQ_C4（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: add_order_size_band

def add_order_size_band(orders):
    """根据商品数量为订单添加有序规模档位。"""
    # 完成提示：
    # 步骤1：创建 orders 的副本。
    # 步骤2：为 quantity 准备边界 [0, 1, 3, np.inf] 和三个新标签。
    # 步骤3：调用 pd.cut，并设置 include_lowest=True、right=True、ordered=True。
    # 步骤4：把结果写入副本的新列 order_size。
    # 易错提醒：这里的分箱列、边界和标签都不同于商品价格示例；
    #          自动测试会检查 1、3 以及边界两侧的值。
    result = ...
    return result

In [22]:
# 测试习题4
testsuite3.test_add_order_size_band(add_order_size_band)

测试失败 test_add_order_size_band: 'ellipsis' object has no attribute 'columns'


## 6. 按键连接：`merge`

显式指定连接键 `on`：`inner` 保留匹配记录，`left` 保留左侧记录，`right` 保留右侧记录，`outer` 保留两侧键的并集；未匹配字段为缺失值。

| `validate` | 唯一性要求 |
|---|---|
| `one_to_one` | 左右键均唯一 |
| `one_to_many` | 左键唯一 |
| `many_to_one` | 右键唯一 |
| `many_to_many` | 不检查唯一性 |

例如订单连接商品表应通常用 `many_to_one`。先选所需列可减少同名列后缀；右键重复时，即使左连接也可能增加行数。同一键左侧2行、右侧3行，会产生6行组合。

```python
import pandas as pd
join_left_demo = pd.DataFrame({'key': ['A', 'A'], 'left_value': [1, 2]})
join_right_demo = pd.DataFrame({'key': ['A', 'A', 'A'], 'right_value': [3, 4, 5]})
print(len(join_left_demo.merge(join_right_demo, on='key', how='inner')))  # 6
```

```python
left_rows_demo = pd.DataFrame({'id': ['A', 'B'], 'quantity': [4, 2]})
right_rows_demo = pd.DataFrame({'id': ['A', 'C'], 'price': [25, 30]})
inner_demo = left_rows_demo.merge(right_rows_demo, on='id', how='inner', validate='many_to_one')
left_demo = left_rows_demo.merge(right_rows_demo, on='id', how='left', validate='many_to_one')
print(inner_demo['id'].tolist())       # ['A']：只保留匹配记录
print(left_demo['price'].isna().tolist())  # [False, True]：B没有匹配价格
```

连接后可用整列公式派生收入：`quantity*unit_price*(1-discount)`。数量4、单价25、折扣比例0.2时收入为80；`round(2)` 保留两位小数。

In [23]:
# 准备用于连接的干净副本
orders_ready = orders.drop_duplicates('order_id', keep='last').copy()
orders_ready['discount'] = orders_ready['discount'].fillna(0.0)
orders_ready['order_date'] = pd.to_datetime(orders_ready['order_date'])
orders_ready['quantity'] = orders_ready['quantity'].astype('int64')

customers_ready = customers_raw.drop_duplicates('customer_id', keep='last').copy()
customers_ready['customer_name'] = customers_ready['customer_name'].str.strip()
customers_ready['city'] = customers_ready['city'].fillna('').str.strip().replace('', 'Unknown').str.title()

sales_example = orders_ready.merge(
    products[['product_id', 'product_name', 'category', 'unit_price']],
    on='product_id', how='left', validate='many_to_one',
).merge(
    customers_ready[['customer_id', 'customer_name', 'city']],
    on='customer_id', how='left', validate='many_to_one',
)
sales_example['revenue'] = (
    sales_example['quantity'] * sales_example['unit_price'] * (1 - sales_example['discount'])
).round(2)
sales_example.head()

,order_id,order_date,customer_id,product_id,quantity,discount,product_name,category,unit_price,customer_name,city,revenue
0,O1001,2026-03-01,C001,P001,2,0.10,Wireless Mouse,Electronics,89,Alice Zhang,Shanghai,160.20
1,O1002,2026-03-01,C002,P002,1,0.00,Mechanical Keyboard,Electronics,329,Bob Li,Beijing,329.00
2,O1003,2026-03-02,C003,P004,5,0.05,Notebook,Stationery,18,Chen Wang,Shanghai,85.50
3,O1004,2026-03-03,C001,P003,1,0.15,USB-C Hub,Electronics,199,Alice Zhang,Shanghai,169.15
4,O1005,2026-03-03,C004,P007,3,0.00,Water Bottle,Home,79,Diana Liu,Guangzhou,237.00


In [24]:
# 左连接保留了未知客户 C999 的订单，未匹配的客户字段为缺失值
sales_example.loc[sales_example['customer_name'].isna()]

,order_id,order_date,customer_id,product_id,quantity,discount,product_name,category,unit_price,customer_name,city,revenue
11,O1012,2026-03-07,C999,P003,1,0.0,USB-C Hub,Electronics,199,NaN,NaN,199.0


## 习题5：表连接

完成 `build_stock_sales_detail(inventory, monthly_sales)`。

数据：`inventory_shanghai.csv` 与 `monthly_sales_wide.csv`。

1. 不修改任何输入 DataFrame；
2. 从 `monthly_sales` 只选择 `product_id` 和 `2026-03`；
3. 以 `product_id` 为键执行内连接，并设置一对一关系校验；
4. 新增 `sales_per_stock = 2026-03销售额 ÷ stock`，保留两位小数；
5. 返回列顺序为 `product_id`、`warehouse`、`stock`、`2026-03`、`sales_per_stock`；
6. 按 `sales_per_stock` 降序、`product_id` 升序排列并重置索引。

In [25]:
# UNQ_C5（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: build_stock_sales_detail

def build_stock_sales_detail(inventory, monthly_sales):
    """连接库存和三月销售数据并计算单位库存销售额。"""
    # 完成提示：
    # 步骤1：只选择 monthly_sales 的 product_id 和 2026-03 两列。
    # 步骤2：使用 merge，以 product_id 为键执行 inner 连接，
    #        并设置 validate='one_to_one'。
    # 步骤3：使用整列运算计算 sales_per_stock，并 round(2)。
    # 步骤4：显式选择五个返回列，再按题目要求复合排序并重置索引。
    # 易错提醒：本题不是 left 连接，也不是 many_to_one；不要连接讲解示例中的客户字段。
    result = ...
    return result

In [26]:
# 测试习题5
testsuite3.test_build_stock_sales_detail(build_stock_sales_detail)

测试失败 test_build_stock_sales_detail: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 7. 分组聚合：`groupby`

按键拆分数据 → 对每组计算 → 合并结果。

| 方法 | 统计内容 |
|---|---|
| `count` | 指定列的非缺失数 |
| `size` | 组内行数，包括缺失 |
| `nunique` | 不同值的数量，默认忽略缺失 |

`as_index=False` 让分组键保留为普通列。命名聚合 `total=('amount','sum')` 中，`total` 是输出列名，`amount` 是输入列，`sum` 是计算方法。也可使用 `mean/min/max` 等聚合。

下面一组有3行、2个非缺失金额和2个不同订单。多列分组默认可产生多级索引，需要普通列时可用 `reset_index()`。

```python
import pandas as pd
groups_demo = pd.DataFrame({
    'city': ['A', 'A', 'A'],
    'order': ['o1', 'o1', 'o2'],
    'amount': [10.0, None, 10.0],
})
summary_demo = groups_demo.groupby('city', as_index=False).agg(
    present=('amount', 'count'),
    rows=('amount', 'size'),
    unique_orders=('order', 'nunique'),
    total=('amount', 'sum'),
)
print(summary_demo[['present', 'rows', 'unique_orders', 'total']].iloc[0].tolist())
# [2.0, 3.0, 2.0, 20.0]
```

In [27]:
category_summary_example = (
    sales_example
    .groupby('category', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('revenue', 'sum'),
    )
)
category_summary_example['total_revenue'] = category_summary_example['total_revenue'].round(2)
category_summary_example = (
    category_summary_example
    .sort_values(
        by=['total_revenue', 'category'],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)
category_summary_example

,category,order_count,total_quantity,total_revenue
0,Electronics,7,8,1406.65
1,Home,2,5,395.00
2,Stationery,3,20,375.25


In [28]:
# 多列分组会形成层次化索引；reset_index 可将索引层级恢复为普通列
sales_example.groupby(['category', 'city'])['revenue'].sum().reset_index().head()

,category,city,revenue
0,Electronics,Beijing,615.10
1,Electronics,Shanghai,592.55
2,Home,Guangzhou,237.00
3,Home,Shanghai,158.00
4,Stationery,Guangzhou,171.00


## 习题6：分组聚合

完成 `summarize_customer_orders(orders)`。

数据：`data/orders.csv`，已读入 `orders`。

1. 不修改输入 DataFrame；
2. 按 `customer_id` 分组；
3. 计算唯一订单数 `order_count`；
4. 计算不同商品数 `product_count`；
5. 计算商品数量合计 `total_quantity`；
6. 计算平均折扣 `average_discount`，并保留三位小数；
7. 返回列顺序为 `customer_id`、`order_count`、`product_count`、`total_quantity`、`average_discount`；
8. 按 `total_quantity` 降序、`customer_id` 升序排列并重置索引。

In [29]:
# UNQ_C6（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: summarize_customer_orders

def summarize_customer_orders(orders):
    """按客户汇总唯一订单、商品、数量和平均折扣。"""
    # 完成提示：
    # 步骤1：按 customer_id 分组，并让分组键保持为普通列。
    # 步骤2：使用命名聚合：order_id 和 product_id 使用 nunique，
    #        quantity 使用 sum，discount 使用 mean。
    # 步骤3：对 average_discount 使用 round(3)。
    # 步骤4：按 total_quantity 降序、customer_id 升序排序并重置索引。
    # 易错提醒：这里汇总的是客户订单，不需要 category、revenue 或销售收入字段。
    result = ...
    return result

In [30]:
# 测试习题6
testsuite3.test_summarize_customer_orders(summarize_customer_orders)

测试失败 test_summarize_customer_orders: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 8. 沿轴拼接：`concat`

`merge` 按键匹配；`concat` 沿轴拼接，不自动去重。

- `axis=0` 追加行，按列名对齐；`axis=1` 追加列，按行索引对齐。
- 纵向拼接时，`join='outer'` 保留列并集，`'inner'` 保留共同列；横向时则作用于行索引。
- `ignore_index=True` 重建拼接轴的连续索引；`keys` 可标记数据来源。
- 缺少的字段会产生缺失值，拼接前应确认字段含义和类型一致。

下面重建了行索引，但两个B仍然保留。

```python
import pandas as pd
batch_a_demo = pd.DataFrame({'id': ['A', 'B']})
batch_b_demo = pd.DataFrame({'id': ['B']})
combined_demo = pd.concat([batch_a_demo, batch_b_demo], axis=0, ignore_index=True)
print(combined_demo['id'].tolist())  # ['A', 'B', 'B']：重复值仍保留
print(combined_demo.index.tolist())  # [0, 1, 2]：只重建行索引
```

In [31]:
all_inventory = pd.concat(
    [inventory_shanghai, inventory_beijing],
    axis='index',
    ignore_index=True,
)
all_inventory

,product_id,warehouse,stock
0,P001,Shanghai,30
1,P002,Shanghai,12
2,P003,Shanghai,8
3,P004,Shanghai,100
4,P005,Shanghai,80
5,P003,Beijing,15
6,P004,Beijing,60
7,P006,Beijing,10
8,P007,Beijing,20
9,P008,Beijing,14


In [32]:
inventory_summary_example = (
    all_inventory
    .groupby('product_id', as_index=False)
    .agg(
        total_stock=('stock', 'sum'),
        warehouse_count=('warehouse', 'nunique'),
    )
    .sort_values(['total_stock', 'product_id'], ascending=[False, True])
    .reset_index(drop=True)
)
inventory_summary_example

,product_id,total_stock,warehouse_count
0,P004,160,2
1,P005,80,1
2,P001,30,1
3,P003,23,2
4,P007,20,1
5,P008,14,1
6,P002,12,1
7,P006,10,1


## 习题7：纵向拼接与汇总

完成 `summarize_order_batches(first_batch, second_batch)`。

输入为两批订单，字段结构与 `orders.csv` 相同。

1. 不修改两个输入 DataFrame；
2. 使用 `pd.concat` 纵向拼接两批订单并重置索引；
3. 按 `customer_id` 分组；
4. 计算唯一订单数 `order_count`；
5. 计算商品数量合计 `total_quantity`；
6. 返回列顺序为 `customer_id`、`order_count`、`total_quantity`；
7. 按 `total_quantity` 降序、`customer_id` 升序排列并重置索引。

In [33]:
# UNQ_C7（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: summarize_order_batches

def summarize_order_batches(first_batch, second_batch):
    """拼接两批订单并按客户汇总。"""
    # 完成提示：
    # 步骤1：使用 pd.concat 纵向拼接两个参数，并设置 ignore_index=True。
    # 步骤2：按 customer_id 分组。
    # 步骤3：使用命名聚合计算 order_id 的 nunique 和 quantity 的 sum。
    # 步骤4：选择并排列三个返回列，再复合排序并重置索引。
    # 易错提醒：本题的数据列来自订单，不再使用 product_id、warehouse 或 stock 汇总库存。
    result = ...
    return result

In [34]:
# 测试习题7
testsuite3.test_summarize_order_batches(summarize_order_batches)

测试失败 test_summarize_order_batches: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 9. 宽表与长表

宽表把月份等类别放在列中；长表把它们放在一列里，每行是一条观察，便于分组和绘图。

| `melt` 参数 | 作用 |
|---|---|
| `id_vars` | 保留的标识列 |
| `value_vars` | 转换的列；省略时取全部非标识列 |
| `var_name` | 存原列名的新列 |
| `value_name` | 存原数值的新列 |

`melt` 返回新表；需要稳定顺序时再排序、重置索引。

反向转换中，`pivot` 要求“行键+列键”唯一；有重复时，先确认业务含义，再用 `pivot_table(aggfunc=...)` 明确按均值、和等聚合。`stack` 把列索引层移到行索引，`unstack` 则相反。

```python
import pandas as pd
wide_demo = pd.DataFrame({'item': ['A', 'B'], 'Mar': [10, 20], 'Apr': [30, 40]})
long_demo = wide_demo.melt(id_vars='item', var_name='month', value_name='sales')
print(long_demo.shape)                # (4, 3)：每个商品每个月一行
print(long_demo.columns.tolist())     # ['item', 'month', 'sales']
```

```python
repeated_demo = pd.DataFrame({
    'item': ['A', 'A', 'B'],
    'month': ['Mar', 'Mar', 'Mar'],
    'sales': [10, 30, 50],
})
try:
    repeated_demo.pivot(index='item', columns='month', values='sales')
except ValueError:
    print('重复组合，pivot不能直接转换')
mean_demo = repeated_demo.pivot_table(index='item', columns='month', values='sales', aggfunc='mean')
print(mean_demo['Mar'].tolist())  # [20.0, 50.0]：A按均值聚合
```

In [35]:
monthly_sales_wide

,product_id,2026-01,2026-02,2026-03
0,P001,1200,1350,1280
1,P002,980,1120,1460
2,P003,1560,1490,1720
3,P004,620,710,680
4,P005,540,590,640
5,P007,810,760,920
6,P008,1340,1410,1530


In [36]:
monthly_sales_long = monthly_sales_wide.melt(
    id_vars='product_id',
    var_name='month',
    value_name='sales',
).sort_values(['product_id', 'month']).reset_index(drop=True)
monthly_sales_long.head(8)

,product_id,month,sales
0,P001,2026-01,1200
1,P001,2026-02,1350
2,P001,2026-03,1280
3,P002,2026-01,980
4,P002,2026-02,1120
5,P002,2026-03,1460
6,P003,2026-01,1560
7,P003,2026-02,1490


In [37]:
# pivot 可以把长表恢复为宽表
monthly_sales_long.pivot(
    index='product_id',
    columns='month',
    values='sales',
).reset_index()

month,product_id,2026-01,2026-02,2026-03
0,P001,1200,1350,1280
1,P002,980,1120,1460
2,P003,1560,1490,1720
3,P004,620,710,680
4,P005,540,590,640
5,P007,810,760,920
6,P008,1340,1410,1530


## 习题8：宽表转长表

完成 `reshape_product_attributes(products)`。

数据：`data/products.csv`，已读入 `products`。

1. 不修改输入 DataFrame；
2. 使用 `melt`，保留 `product_id` 和 `unit_price` 作为标识列；
3. 只转换 `product_name`、`category`、`status` 三列；
4. 原列名保存到 `attribute`，对应的值保存到 `value`；
5. 返回列顺序为 `product_id`、`unit_price`、`attribute`、`value`；
6. 按 `product_id`、`attribute` 升序排列并重置索引。

In [38]:
# UNQ_C8（不要修改这两行注释，否则该习题无法自动评分）
# GRADED FUNCTION: reshape_product_attributes

def reshape_product_attributes(products):
    """把商品的多个属性列转换成长表。"""
    # 完成提示：
    # 步骤1：调用 melt，把 product_id、unit_price 作为 id_vars。
    # 步骤2：把 product_name、category、status 作为 value_vars。
    # 步骤3：设置 var_name='attribute'、value_name='value'。
    # 步骤4：显式选择四个返回列，按 product_id、attribute 排序并重置索引。
    # 易错提醒：本题转换的是商品属性，不要使用讲解示例中的 month 或 sales 列名。
    result = ...
    return result

In [39]:
# 测试习题8
testsuite3.test_reshape_product_attributes(reshape_product_attributes)

测试失败 test_reshape_product_attributes: DataFrame Expected type <class 'pandas.core.frame.DataFrame'>, found <class 'ellipsis'> instead


## 10. 实验讨论

请结合代码和运行结果讨论：

1. 为什么 pandas 按索引自动对齐既方便又可能产生意外的缺失值？
   
2. 删除缺失值、填补缺失值分别适合什么情况？不恰当处理会造成什么偏差？
   
3. `merge` 和 `concat` 的用途有什么本质区别？为什么表连接时要检查键的唯一性？
   
4. 长表和宽表分别适合哪些任务？请各举一个例子。

## 11. 生成并提交实验报告

1. 保存 Notebook，运行下方合并单元格，依次完成评分、生成 PDF 和提交。
2. 生成失败会立即停止；提交成功会显示回执编号。若提交失败，本地 PDF 会保留，检查网络或服务器后重试。
3. 非实验室环境补交时，使用[课程收集表](https://docs.qq.com/form/page/DWVhDZ3JPdGRja3RP)。补交按课程规定扣分；需要更正时修改原提交，避免重复提交。

In [ ]:
# 保存 Notebook 后运行本单元格：评分 → 生成 PDF → 提交到 LabDrop
import importlib
import notebook2pdf
import report_submission

importlib.reload(notebook2pdf)
importlib.reload(report_submission)
from report_submission import submit_pdf_report
receipt = None  # 清除上次运行的回执，避免把旧结果当作本次成功

print("[1/3] 正在计算实验成绩……")
stu_grade = testsuite3.grade_all_tests(notebook_file)
print(f"评分完成：{stu_grade} 分。")

print("[2/3] 正在生成 PDF 报告……")
pdf_name = f"{stu_info['class_id']}-{stu_info['student_id']}-{stu_info['name']}-实验报告3-{stu_grade}.pdf"
# 生成失败会抛出异常，下面的提交代码不会执行。
pdf_file = notebook2pdf.convert_notebook_to_webpdf(notebook_file, pdf_name)

print("[3/3] 正在提交 PDF 报告到 LabDrop……")
try:
    receipt = submit_pdf_report(pdf_file, stu_info, experiment_number=3)
except Exception as error:
    print(f"报告提交失败；已生成的 PDF 保留在：{pdf_file}")
    print(f"原因：{error}")
    print("请检查网络或服务器后重试；本次没有收到提交成功回执。")
    raise

print("生成与提交流程完成，请保留本地 PDF 和提交回执。")
receipt

## 参考资料

- Wes McKinney, *Python for Data Analysis, 3rd Edition*：第 5 章 Getting Started with pandas、第 7 章 Data Cleaning and Preparation、第 8 章 Data Wrangling: Join, Combine, and Reshape。
- pandas 官方文档：https://pandas.pydata.org/docs/